# Weakly Connected Components with Neptune Analytics

This notebook demonstrates the Weakly Connected Components (WCC) algorithm using nx-neptune.
WCC finds groups of nodes where every node is reachable from every other node when edge directions
are ignored. This is useful for identifying disconnected subgraphs, isolated clusters, and
understanding the overall connectivity structure of a graph.

## Setup and Imports

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import os
import requests
import pandas as pd

import networkx as nx
from nx_neptune import NeptuneGraph
from nx_neptune.clients import Node
from nx_neptune.utils.utils import get_stdout_logger

In [ ]:
logger = get_stdout_logger(__name__,[
                    'nx_neptune.algorithms.communities.wcc',
                    'nx_neptune.na_graph', 'nx_neptune.utils.decorators',
                    'nx_neptune.instance_management',__name__])

# Ignore cache warnings
nx.config.warnings_to_ignore.add("cache")

## Check for Neptune Analytics Graph ID

In [ ]:
# Read and load graphId from environment variable
graph_id = os.getenv('NETWORKX_GRAPH_ID')

# If not set, you can set it here
if not graph_id:
    # Uncomment and set your Graph ID
    # %env NETWORKX_GRAPH_ID=your-neptune-analytics-graph-id
    # graph_id = os.getenv('NETWORKX_GRAPH_ID')
    print("Warning: Environment Variable NETWORKX_GRAPH_ID is not defined")
    print("You can set it using: %env NETWORKX_GRAPH_ID=your-neptune-analytics-graph-id")
else:
    print(f"Using Neptune Analytics Graph ID: {graph_id}")

## Download and configure Air route dataset

In [ ]:
# Download routes data
routes_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"
routes_file = "resources/notebook_test_data_routes.dat"

# Ensure the directory exists
os.makedirs(os.path.dirname(routes_file), exist_ok=True)

# Download only if file doesn't exist
if not os.path.isfile(routes_file):
    with open(routes_file, "wb") as f:
        f.write(requests.get(routes_url).content)

cols = [
    "airline", "airline_id", "source_airport", "source_airport_id",
    "dest_airport", "dest_airport_id", "codeshare", "stops", "equipment",
]
routes = pd.read_csv(routes_file, names=cols)
routes = routes[["source_airport", "dest_airport"]].dropna()

air_route_graph = nx.from_pandas_edgelist(
    routes, source="source_airport", target="dest_airport",
    create_using=nx.DiGraph()
)
print(f"Graph loaded: {air_route_graph.number_of_nodes()} nodes, {air_route_graph.number_of_edges()} edges")

## Example 1: Basic WCC detection

Find all weakly connected components in the graph. The result is a list of sets, where each set contains the node IDs belonging to the same component.

In [ ]:
result = nx.weakly_connected_components(air_route_graph, backend="neptune")

# Sort by size, show top 5 components
sorted_components = sorted(result, key=len, reverse=True)
logger.info(f"Total components found: {len(sorted_components)}")
for i, component in enumerate(sorted_components[:5], 1):
    sample = list(component)[:3]
    logger.info(f"Component {i} - Size:{len(component)} - Sample: {sample}...")

## Example 2: WCC with edge label filtering

You can filter which edge labels to consider during traversal.

In [ ]:
result = nx.weakly_connected_components(
    air_route_graph,
    backend="neptune",
    edge_labels=["route"]
)

sorted_components = sorted(result, key=len, reverse=True)
logger.info(f"Components with 'route' edges only: {len(sorted_components)}")
for i, component in enumerate(sorted_components[:3], 1):
    logger.info(f"Component {i} - Size: {len(component)}")

## Example 3: Mutation (write_property)

Store the component IDs as node properties for later queries.

In [ ]:
nx.weakly_connected_components(air_route_graph, backend="neptune", write_property="wccid")

"""List 10 nodes to verify the property was written"""
nx_graph = NeptuneGraph.from_config(graph=air_route_graph)
for item in nx_graph.get_all_nodes()[:10]:
    logger.info(Node.from_neptune_response(item))